## A pattern for solving for blobby regional assignments fitting a distribution

In [ ]:
import base64
import json
import pyrender
import random
import numpy as np
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from scipy.ndimage import gaussian_filter as blur
from typing import Callable, List, Optional, Tuple, Union

from perlin_numpy import (
    generate_fractal_noise_2d,
    generate_fractal_noise_3d,
    generate_perlin_noise_2d,
    generate_perlin_noise_3d
)

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
Noise = Union["SimpleNoise", "RidgedNoise", "SummedNoise"]

In [ ]:
@dataclass
class SimpleNoise:
    size: int
    octaves: int
    persistence: float
        
    def __call__(self, shape=(512, 512)):
        return generate_fractal_noise_2d(
            shape=shape,
            res=(shape[0] // self.size, shape[1] // self.size),
            octaves=self.octaves,
            persistence=self.persistence,
            lacunarity=2,
        )

In [ ]:
@dataclass
class RidgedNoise:
    size: int
    octaves: int
    persistence: float
        
    def __call__(self, shape=(512, 512)):
        noise = generate_fractal_noise_2d(
            shape=shape,
            res=(shape[0] // self.size, shape[1] // self.size),
            octaves=self.octaves,
            persistence=self.persistence,
            lacunarity=2,
        )
        noise = -np.abs(noise)
        return noise - noise.mean()

In [ ]:
@dataclass
class SummedNoise:
    lhs: Noise
    rhs: Noise
        
    def __call__(self, shape=(512, 512)):
        return self.lhs(shape) + self.rhs(shape)

In [ ]:
@dataclass
class ShiftedNoise:
    shift: Noise
    noise: Noise
        
    def __call__(self, shape=(512, 512)):
        return self.shift + self.noise(shape)

In [ ]:
@dataclass
class ScaledNoise:
    scale: Noise
    noise: Noise
        
    def __call__(self, shape=(512, 512)):
        return self.scale * self.noise(shape)

In [ ]:
@dataclass
class ExpNoise:
    noise: Noise
        
    def __call__(self, shape=(512, 512)):
        return np.exp(self.noise(shape))

In [ ]:
@dataclass
class InvertedNoise:
    noise: Noise
        
    def __call__(self, shape=(512, 512)):
        return -self.noise(shape)

In [ ]:
noise = SimpleNoise(128, 4, 0.4)()**2
heatmap_to_image((noise - noise.min()) / (noise.max() - noise.min()))

In [ ]:
noise = (SimpleNoise(128, 4, 0.4)() > 0.3).astype(float)
heatmap_to_image((noise - noise.min()) / (noise.max() - noise.min()))

In [ ]:
noise = (RidgedNoise(256, 2, 0.4)() > 0.15).astype(float)
heatmap_to_image((noise - noise.min()) / (noise.max() - noise.min()))

In [ ]:
noise = SimpleNoise(128, 6, 0.6)
noise = (noise() > 0.12).astype(float)
heatmap_to_image((noise - noise.min()) / (noise.max() - noise.min()))

In [ ]:
# TODO: Add a hierachy (i.e. concept of parent region).

@dataclass
class Region:
    name: str
    rarity: float
    noise: Callable[Tuple[int, int], np.ndarray]
    color: Tuple[int, int, int]

In [ ]:
regions = [
    Region(
        name="river",
        rarity=0.05,
        noise=RidgedNoise(256, 2, 0.5),
        color=[50, 100, 75],
    ),
    Region(
        name="mountain",
        rarity=0.2,
        noise=SimpleNoise(256, 6, 0.4),
        color=[50, 100, 75],
    ),
    Region(
        name="forest",
        rarity=0.3,
        noise=SimpleNoise(128, 4, 0.4),
        color=[50, 100, 75],
    ),
    Region(
        name="meadow",
        rarity=0.2,
        noise=SimpleNoise(64, 4, 0.3),
        color=[50, 100, 75],
    ),
    Region(
        name="cotton",
        rarity=0.01,
        noise=SimpleNoise(32, 2, 0.5),
        color=[50, 100, 75],
    ),
]

In [ ]:
thresholds = [ 0 ] * len(regions)

### Solve for the thresholds

In [ ]:
%%time

steps = 10
shape = (1024, 1024)

for i, region in enumerate(regions):
    thresholds[i] = 0
    
    for step in range(steps):
        
        mask = np.ones(shape, dtype=bool)
        for j, prior in enumerate(regions[:i]):
            mask &= np.logical_not(prior.noise(shape) > thresholds[j])

        nm = region.noise(shape)
        nm[np.logical_not(mask)] = float("-inf")
        
        thresholds[i] += np.quantile(nm, 1 - region.rarity)
        
    thresholds[i] /= steps
    
    actual = np.logical_and(mask, nm > thresholds[i]).mean()
    print(f"region: {i}, threshold: {thresholds[i]:.3f} target: {region.rarity}, actual: {actual}")

### Generate an assignment

In [ ]:
shape = (512, 512)

assignment = len(regions) * np.ones(shape, dtype=int)
for i, region in enumerate(regions):
    nm = region.noise(shape)
    mask = np.logical_and(assignment == len(regions), nm > thresholds[i])
    assignment[mask] = i

In [ ]:
palette = np.random.randint(0, 255, size=(assignment.max() + 1, 3), dtype=np.uint8)
Image.fromarray(palette[assignment])

### Generate height maps from the assignment

In [ ]:
harmonics = [
    generate_perlin_noise_2d(
        shape=(512, 512),
        res=(2**i, 2**i)
    )
    for i in range(0, 9)
]

In [ ]:
def combine_harmonics(weights):
    assert len(weights) == len(harmonics)
    return np.stack([w * h for w, h in zip(weights, harmonics)]).sum(axis=0)

In [ ]:
def generate_heights(weights, mean, std):
    ret = combine_harmonics(weights)
    ret = (ret - ret.mean()) / ret.std()
    ret = std * (ret + mean)
    return ret

In [ ]:
class NoiseWeights:
    BASE = [32, 4, 4, 2, 0, 0, 0, 0.0, 0.0]
    CRAZY = [0, 0, 0, 2, 1, 1, 1, 1, 1]
    ROUGH = [0, 0, 32, 16, 4, 4, 2, 1, 1]
    HILLS = [0, 0, 16, 8, 4, 1, 0, 0.0, 0.0]
    SMOOTH = [0, 0, 0, 0.3, 0.1, 0.0, 0.0, 0.0, 0.0]
    JAGGED = [0, 0, 0, 0.3, 0.1, 0.0, 0.0, 0.2, 0.1]
    FLAT = [0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001]

In [ ]:
noise = generate_heights(NoiseWeights.ROUGH, 0.0, 8)
heatmap_to_image((noise - noise.min()) / (noise.max() - noise.min()))

In [ ]:
@dataclass
class RegionHeights:
    name: str
    weights: List[float]
    radius: float
    mean: float
    std: float

In [ ]:
region_heights = [
    RegionHeights(
        name="river",
        weights=NoiseWeights.SMOOTH,
        radius=0.1,
        mean=-4,
        std=2,
    ),
    RegionHeights(
        name="mountain",
        weights=NoiseWeights.ROUGH,
        radius=8,
        mean=2,
        std=16,
    ),
    RegionHeights(
        name="forest",
        weights=NoiseWeights.SMOOTH,
        radius=4,
        mean=0,
        std=1,
    ),
    RegionHeights(
        name="meadow",
        weights=NoiseWeights.SMOOTH,
        radius=4,
        mean=0,
        std=0.1,
    ),
    RegionHeights(
        name="cotton",
        weights=NoiseWeights.SMOOTH,
        radius=0,
        mean=0,
        std=0.1,
    ),
]

In [ ]:
def inclusive_blur(mask, radius):
    ret = blur(mask.astype(float), radius)
    lo = ret[np.logical_not(mask)].max()
    ret[np.logical_not(mask)] = 0.0
    return np.clip(0.5 * (ret - lo) / (1 - lo) + 0.5, 0, 1)

In [ ]:
%%time

hm = combine_harmonics(NoiseWeights.BASE)

for i, heights in enumerate(region_heights):
    update = generate_heights(heights.weights, heights.mean, heights.std)
    weight = inclusive_blur(assignment == i, heights.radius)
    hm += weight * update

In [ ]:
heatmap_to_image((hm - hm.min()) / (hm.max() - hm.min()))

### Render height map in 3D

In [ ]:
hm -= hm.min() - 1
hm = hm.astype(int)

In [ ]:
voxels = np.zeros(shape=(hm.shape[0], 300, hm.shape[1]), dtype=np.uint32)

In [ ]:
%%time

for z in range(hm.shape[0]):
    for x in range(hm.shape[1]):
        h = hm[z, x]
        voxels[z, 0:h, x] = 0xffffffff

In [ ]:
%%time 

mesh = voxeloo.voxels.voxels_to_mesh(voxels)

In [ ]:
scene = pyrender.Scene()
scene.add(
    pyrender.Mesh.from_trimesh(
        trimesh.Trimesh(
            vertices=mesh.vertices[:, 0:3],
            faces=mesh.triangles,
        ),
        smooth=False,
    )
)
pyrender.Viewer(scene, use_raymond_lighting=True)